<a href="https://colab.research.google.com/github/MarinaShtefan/HR-Policy-Project/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82_%D0%BA%D0%BE%D0%BC%D0%B0%D0%BD%D0%B4%D1%8B_3_%D0%9F%D1%80%D0%B5%D0%B4%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Предобработка данных: HR-аналитика (кейс 4)

## Датасет
Работаем с данными о сотрудниках компании (HR Analytics).

Используемые файлы:
- `Employee.csv` — основная таблица сотрудников (ID, пол, возраст, отдел, зарплата, дата приема, текучесть и пр.);
- `PerformanceRating.csv` — таблица оценок производительности и удовлетворённости;
- `EducationLevel.csv` — справочник уровней образования;
- `RatingLevel.csv` — справочник уровней рейтинга;
- `SatisfiedLevel.csv` — справочник уровней удовлетворенности.

## Цель этапа
Провести предобработку: привести названия колонок к стандарту, преобразовать даты, обработать пропуски, объединить таблицы и создать новые признаки для последующего анализа текучести.

## Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Загрузка исходных данных

Загружаем все CSV-файлы кейса.

In [ ]:
education = pd.read_csv('/content/drive/MyDrive/Учеба/Проект команды 3/EducationLevel.csv')
employee = pd.read_csv('/content/drive/MyDrive/Учеба/Проект команды 3/Employee.csv')
performance = pd.read_csv('/content/drive/MyDrive/Учеба/Проект команды 3/PerformanceRating.csv')
rating = pd.read_csv('/content/drive/MyDrive/Учеба/Проект команды 3/RatingLevel.csv')
satisfied = pd.read_csv('/content/drive/MyDrive/Учеба/Проект команды 3/SatisfiedLevel.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Учеба/Проект команды 3/EducationLevel.csv'

## Первичный осмотр данных

Посмотрим структуру, типы и первые строки каждой таблицы: `.info()`, `.head()`, `.shape`.

In [ ]:
education.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   EducationLevelID  5 non-null      int64 
 1   EducationLevel    5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
education.head()

,EducationLevelID,EducationLevel
0,1,No Formal Qualifications
1,2,High School
2,3,Bachelors
3,4,Masters
4,5,Doctorate


In [ ]:
education.shape

(5, 2)

***Вывод**:  данные чистые, полные, без дублей и пропусков, представляют собой полный справочник из 5 категорий образования.*

In [ ]:
employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   EmployeeID               1470 non-null   object
 1   FirstName                1470 non-null   object
 2   LastName                 1470 non-null   object
 3   Gender                   1470 non-null   object
 4   Age                      1470 non-null   int64 
 5   BusinessTravel           1470 non-null   object
 6   Department               1470 non-null   object
 7   DistanceFromHome (KM)    1470 non-null   int64 
 8   State                    1470 non-null   object
 9   Ethnicity                1470 non-null   object
 10  Education                1470 non-null   int64 
 11  EducationField           1470 non-null   object
 12  JobRole                  1470 non-null   object
 13  MaritalStatus            1470 non-null   object
 14  Salary                   1470 non-null  

(1470, 23)

In [ ]:
employee.head()

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,...,MaritalStatus,Salary,StockOptionLevel,OverTime,HireDate,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,Divorced,102059,1,No,2012-01-03,No,10,4,9,7
1,CBCB-9C9D,Leonerd,Aland,Male,38,Some Travel,Sales,23,CA,White,...,Single,157718,0,Yes,2012-01-04,No,10,6,10,0
2,95D7-1CE9,Ahmed,Sykes,Male,43,Some Travel,Human Resources,29,CA,Asian or Asian American,...,Married,309964,1,No,2012-01-04,No,10,6,10,8
3,47A0-559B,Ermentrude,Berrie,Non-Binary,39,Some Travel,Technology,12,IL,White,...,Married,293132,0,No,2012-01-05,No,10,10,10,0
4,42CC-040A,Stace,Savege,Female,29,Some Travel,Human Resources,29,CA,White,...,Single,49606,0,No,2012-01-05,Yes,6,1,1,6


In [ ]:
employee.shape

(1470, 23)

***Вывод**: данные таблицы чистые, полные, без дублей и пропусков, но требуют базовой предобработки.*

Ключевые наблюдения:
1. Gender включает значение Non-Binary — данные учитывают больше двух категорий пола.
2. Salary сильно разбросан: от ~49 606 до ~309 964 в первых пяти строках.
3. Attrition уже встречается как Yes (строка 4) и No (остальные) — нужно проверить распределение на всем датасете.
4. HireDate представлен строками (2012-01-03), а не датами — потребуется приведение к datetime.

In [ ]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6709 entries, 0 to 6708
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   PerformanceID                    6709 non-null   object
 1   EmployeeID                       6709 non-null   object
 2   ReviewDate                       6709 non-null   object
 3   EnvironmentSatisfaction          6709 non-null   int64 
 4   JobSatisfaction                  6709 non-null   int64 
 5   RelationshipSatisfaction         6709 non-null   int64 
 6   TrainingOpportunitiesWithinYear  6709 non-null   int64 
 7   TrainingOpportunitiesTaken       6709 non-null   int64 
 8   WorkLifeBalance                  6709 non-null   int64 
 9   SelfRating                       6709 non-null   int64 
 10  ManagerRating                    6709 non-null   int64 
dtypes: int64(8), object(3)
memory usage: 576.7+ KB


In [ ]:
performance.head()

,PerformanceID,EmployeeID,ReviewDate,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating
0,PR01,79F7-78EC,1/2/2013,5,4,5,1,0,4,4,4
1,PR02,B61E-0F26,1/3/2013,5,4,4,1,3,4,4,3
2,PR03,F5E3-48BB,1/3/2013,3,4,5,3,2,3,5,4
3,PR04,0678-748A,1/4/2013,5,3,2,2,0,2,3,2
4,PR05,541F-3E19,1/4/2013,5,2,3,1,0,4,4,3


In [ ]:
performance.shape

(6709, 11)

***Вывод**: данные таблицы чистые, полные, без дублей и пропусков, но требуют предобработки.*

Ключевые наблюдения:

1. ReviewDate хранится как текст (object) — требует приведения к datetime.
2. Есть пара SelfRating / ManagerRating — позволяет сравнивать самооценку с оценкой руководителя (выявить завышенные/заниженные самооценки).

In [ ]:
rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   RatingID     5 non-null      int64 
 1   RatingLevel  5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
rating.head()

,RatingID,RatingLevel
0,1,Unacceptable
1,2,Needs Improvement
2,3,Meets Expectation
3,4,Exceeds Expectation
4,5,Above and Beyond


In [ ]:
rating.shape

(5, 2)

***Вывод**:  данные чистые, полные, без дублей и пропусков, представляют собой полный справочник из 5 категорий оценки эффективности.*

In [ ]:
satisfied.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SatisfactionID     5 non-null      int64 
 1   SatisfactionLevel  5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


In [ ]:
satisfied.head()

,SatisfactionID,SatisfactionLevel
0,1,Very Dissatisfied
1,2,Dissatisfied
2,3,Neutral
3,4,Satisfied
4,5,Very Satisfied


In [ ]:
satisfied.shape

(5, 2)

***Вывод**:  данные чистые, полные, без дублей и пропусков, представляют собой полный справочник из 5 категорий удовлетворенности.*

## Преобразование названий колонок

Приведем имена колонок к нижнему регистру, заменим пробелы и слэши на нижнее подчеркивание, создадим новые переменные.

In [ ]:
def clean_columns(df):
    return df.rename(columns=lambda name: name.lower().replace(' ', '_').replace('/', '_').replace('-', '_'))

education_clean = clean_columns(education)
employee_clean = clean_columns(employee)
performance_clean = clean_columns(performance)
rating_clean = clean_columns(rating)
satisfied_clean = clean_columns(satisfied)

In [ ]:
print(education_clean.columns.tolist())

['educationlevelid', 'educationlevel']


In [ ]:
print(employee_clean.columns.tolist())

['employeeid', 'firstname', 'lastname', 'gender', 'age', 'businesstravel', 'department', 'distancefromhome_(km)', 'state', 'ethnicity', 'education', 'educationfield', 'jobrole', 'maritalstatus', 'salary', 'stockoptionlevel', 'overtime', 'hiredate', 'attrition', 'yearsatcompany', 'yearsinmostrecentrole', 'yearssincelastpromotion', 'yearswithcurrmanager']


In [ ]:
print(performance_clean.columns.tolist())

['performanceid', 'employeeid', 'reviewdate', 'environmentsatisfaction', 'jobsatisfaction', 'relationshipsatisfaction', 'trainingopportunitieswithinyear', 'trainingopportunitiestaken', 'worklifebalance', 'selfrating', 'managerrating']


In [ ]:
print(rating_clean.columns.tolist())

['ratingid', 'ratinglevel']


In [ ]:
print(satisfied_clean.columns.tolist())

['satisfactionid', 'satisfactionlevel']


## Преобразование дат

Переведем текстовые даты в тип `datetime64`.

In [ ]:
employee_clean['hiredate'] = pd.to_datetime(employee_clean['hiredate'], format='%Y-%m-%d')
performance_clean['reviewdate'] = pd.to_datetime(performance_clean['reviewdate'], format='%m/%d/%Y')

In [ ]:
print(employee_clean[['employeeid', 'hiredate']].head())

  employeeid   hiredate
0  3012-1A41 2012-01-03
1  CBCB-9C9D 2012-01-04
2  95D7-1CE9 2012-01-04
3  47A0-559B 2012-01-05
4  42CC-040A 2012-01-05


In [ ]:
print(performance_clean[['performanceid', 'reviewdate']].head())

  performanceid reviewdate
0          PR01 2013-01-02
1          PR02 2013-01-03
2          PR03 2013-01-03
3          PR04 2013-01-04
4          PR05 2013-01-04


In [ ]:
print(employee_clean.dtypes[['hiredate']])

hiredate    datetime64[ns]
dtype: object


In [ ]:
print(performance_clean.dtypes[['reviewdate']])

reviewdate    datetime64[ns]
dtype: object


## Объединение таблиц

Соединим основную таблицу сотрудников с таблицей оценок и справочниками. Также добавим текстовые метки из справочников вместо числовых кодов.

In [ ]:
df = employee_clean.merge(
    performance_clean,
    on='employeeid',
    how='left',
    validate='1:m'
)

df = df.merge(
    education_clean,
    left_on='education',
    right_on='educationlevelid',
    how='left'
)

df = df.merge(
    rating_clean,
    left_on='selfrating',
    right_on='ratingid',
    how='left'
)

df = df.merge(
    satisfied_clean,
    left_on='jobsatisfaction',
    right_on='satisfactionid',
    how='left'
)

print('После объединения:', df.shape)

После объединения: (6899, 39)


In [ ]:
print(df.head())

  employeeid firstname lastname  gender  age businesstravel department  \
0  3012-1A41  Leonelle    Simco  Female   30    Some Travel      Sales   
1  3012-1A41  Leonelle    Simco  Female   30    Some Travel      Sales   
2  3012-1A41  Leonelle    Simco  Female   30    Some Travel      Sales   
3  3012-1A41  Leonelle    Simco  Female   30    Some Travel      Sales   
4  3012-1A41  Leonelle    Simco  Female   30    Some Travel      Sales   

   distancefromhome_(km) state ethnicity  ...  trainingopportunitiestaken  \
0                     27    IL     White  ...                         0.0   
1                     27    IL     White  ...                         1.0   
2                     27    IL     White  ...                         0.0   
3                     27    IL     White  ...                         1.0   
4                     27    IL     White  ...                         0.0   

  worklifebalance selfrating managerrating  educationlevelid  educationlevel  \
0           

In [ ]:
print(df.dtypes)

employeeid                                 object
firstname                                  object
lastname                                   object
gender                                     object
age                                         int64
businesstravel                             object
department                                 object
distancefromhome_(km)                       int64
state                                      object
ethnicity                                  object
education                                   int64
educationfield                             object
jobrole                                    object
maritalstatus                              object
salary                                      int64
stockoptionlevel                            int64
overtime                                   object
hiredate                           datetime64[ns]
attrition                                  object
yearsatcompany                              int64


## Создание новых признаков

Добавим производные переменные, которые пригодятся для анализа текучести:
- `tenure_years` — стаж в компании на текущую дату;
- `age_group` — возрастные группы;
- `salary_band` — зарплатные диапазоны;
- `satisfaction_score` — обобщенный индекс удовлетворенности (среднее по трем шкалам);
- `attrition_flag` — бинарная целевая переменная (1 — уволился, 0 — остался).

In [ ]:
from datetime import datetime

today = pd.Timestamp(datetime.now())

df['tenure_years'] = ((today - df['hiredate']).dt.days / 365.25).round(1)

bins_age = [0, 30, 40, 50, 100]
labels_age = ['до 30', '30-40', '40-50', '50+']
df['age_group'] = pd.cut(df['age'], bins=bins_age, labels=labels_age, right=False)

bins_salary = [0, 50000, 100000, 150000, np.inf]
labels_salary = ['низкая', 'средняя', 'выше среднего', 'высокая']
df['salary_band'] = pd.cut(df['salary'], bins=bins_salary, labels=labels_salary)

sat_cols = ['environmentsatisfaction', 'jobsatisfaction', 'relationshipsatisfaction']
df['satisfaction_score'] = df[sat_cols].mean(axis=1).round(2)

df['attrition_flag'] = (df['attrition'] == 'Yes').astype(int)

print(df[['employeeid', 'age', 'age_group', 'salary', 'salary_band', 'tenure_years', 'satisfaction_score', 'attrition_flag']].head(10))

  employeeid  age age_group  salary    salary_band  tenure_years  \
0  3012-1A41   30     30-40  102059  выше среднего          14.9   
1  3012-1A41   30     30-40  102059  выше среднего          14.9   
2  3012-1A41   30     30-40  102059  выше среднего          14.9   
3  3012-1A41   30     30-40  102059  выше среднего          14.9   
4  3012-1A41   30     30-40  102059  выше среднего          14.9   
5  3012-1A41   30     30-40  102059  выше среднего          14.9   
6  3012-1A41   30     30-40  102059  выше среднего          14.9   
7  3012-1A41   30     30-40  102059  выше среднего          14.9   
8  3012-1A41   30     30-40  102059  выше среднего          14.9   
9  CBCB-9C9D   38     30-40  157718        высокая          14.9   

   satisfaction_score  attrition_flag  
0                2.67               0  
1                4.33               0  
2                4.67               0  
3                2.00               0  
4                3.00               0  
5          

## Сохранение очищенного датасета

Сохраним итоговый табличный файл для следующих этапов анализа и визуализации.

In [ ]:
OUTPUT_PATH = '/content/drive/MyDrive/Учеба/Проект команды 3/Employee_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print('Сохранено:', OUTPUT_PATH)
print('Итоговая размерность:', df.shape)

Сохранено: /content/drive/MyDrive/Учеба/Проект команды 3/Employee_cleaned.csv
Итоговая размерность: (6899, 44)
